In [1]:
import os
import shutil
import xml.etree.ElementTree as ET

VOC_ROOT = "./data/VOCdevkit/VOC2012"
OUTPUT_ROOT = "./VOC_YOLO"

VOC_CLASSES = {
    "aeroplane": 0,
    "bicycle": 1,
    "bird": 2,
    "boat": 3,
    "bottle": 4,
    "bus": 5,
    "car": 6,
    "cat": 7,
    "chair": 8,
    "cow": 9,
    "diningtable": 10,
    "dog": 11,
    "horse": 12,
    "motorbike": 13,
    "person": 14,
    "pottedplant": 15,
    "sheep": 16,
    "sofa": 17,
    "train": 18,
    "tvmonitor": 19,
}


def convert_box(size, box):
    """
    VOC -> YOLO

    Input:
        size = (width, height)

        box = (xmin, ymin, xmax, ymax)

    Returns:
        x_center, y_center, width, height
        normalized to [0,1]
    """

    w, h = size

    xmin, ymin, xmax, ymax = box

    box_width = xmax - xmin
    box_height = ymax - ymin

    x_center = xmin + box_width / 2
    y_center = ymin + box_height / 2

    return (
        x_center / w,
        y_center / h,
        box_width / w,
        box_height / h,
    )


def convert_split(split):

    image_set_file = os.path.join(
        VOC_ROOT,
        "ImageSets",
        "Main",
        f"{split}.txt"
    )

    output_images = os.path.join(
        OUTPUT_ROOT,
        "images",
        split,
    )

    output_labels = os.path.join(
        OUTPUT_ROOT,
        "labels",
        split,
    )

    os.makedirs(output_images, exist_ok=True)
    os.makedirs(output_labels, exist_ok=True)

    with open(image_set_file) as f:
        image_ids = [line.strip() for line in f]

    print(f"{split}: {len(image_ids)} images")

    for image_id in image_ids:

        xml_path = os.path.join(
            VOC_ROOT,
            "Annotations",
            image_id + ".xml"
        )

        image_path = os.path.join(
            VOC_ROOT,
            "JPEGImages",
            image_id + ".jpg"
        )

        tree = ET.parse(xml_path)
        root = tree.getroot()

        width = int(root.find("size/width").text)
        height = int(root.find("size/height").text)

        label_path = os.path.join(
            output_labels,
            image_id + ".txt"
        )

        with open(label_path, "w") as label_file:

            for obj in root.findall("object"):

                difficult = int(obj.find("difficult").text)

                if difficult:
                    continue

                cls = obj.find("name").text

                if cls not in VOC_CLASSES:
                    continue

                cls_id = VOC_CLASSES[cls]

                bbox = obj.find("bndbox")

                xmin = float(bbox.find("xmin").text)
                ymin = float(bbox.find("ymin").text)
                xmax = float(bbox.find("xmax").text)
                ymax = float(bbox.find("ymax").text)

                x, y, w, h = convert_box(
                    (width, height),
                    (xmin, ymin, xmax, ymax)
                )

                label_file.write(
                    f"{cls_id} "
                    f"{x:.6f} "
                    f"{y:.6f} "
                    f"{w:.6f} "
                    f"{h:.6f}\n"
                )

        shutil.copy(image_path, output_images)


convert_split("train")
convert_split("val")

print("Finished!")

train: 5717 images


KeyboardInterrupt: 

In [3]:
from ultralytics import YOLO

In [4]:
model = YOLO("yolov8n.pt")

results = model.train(
    data="./VOC_YOLO/dataset.yaml",
    epochs=15,
    imgsz=640,
    batch=8,
    workers=0,
)


Ultralytics 8.4.102  Python-3.13.14 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070, 12281MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./VOC_YOLO/dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overl

In [5]:
metrics = model.val()

Ultralytics 8.4.102  Python-3.13.14 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070, 12281MiB)
Model summary (fused): 73 layers, 3,009,548 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1005.8298.2 MB/s, size: 102.3 KB)
val: Scanning F:\programming\Python\Deep Learning and LLMs\Pascal-VOC\VOC_YOLO\labels\val.cache... 5823 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5823/5823 2.0Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 364/364 8.2it/s 44.5s0.1ss
                   all       5823      13841      0.777      0.661      0.738      0.537
             aeroplane        343        433      0.864      0.732      0.826      0.631
               bicycle        284        358      0.815      0.729      0.798      0.586
                  bird        370        559      0.812      0.619      0.735      0.517
                  boat        248        424      0.687      0.493   

In [ ]:
results[0].show()